# 과제 3. STL-10 데이터셋을 이용한 CNN 분류 실험

`im2col`, `Convolution`, `Pooling` 계층 구현을 바탕으로 STL-10 이미지 분류 문제에 적용해보는 것이 목표이다.  
MNIST는 28×28 흑백 이미지였지만, STL-10은 96×96 RGB 이미지라서 입력 크기와 채널 수가 다르다.


1. STL-10 데이터셋 불러오기 및 학습/검증/시험 데이터 분리  
2. 17강 코드 기반 직접 구현 CNN 학습  
3. 같은 구조의 PyTorch 모델 학습  
4. 같은 구조의 TensorFlow 모델 학습  
5. 세 모델의 성능, 파라미터 수, FLOPs 비교  
6. 학습된 CNN의 convolution 계층별 특징맵 시각화

In [ ]:
# ============================================================
# 기본 설정
# ============================================================

import os
import random
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

IMG_SIZE = 96
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 50

print("IMG_SIZE:", IMG_SIZE)
print("EPOCHS:", EPOCHS)

In [ ]:
# ============================================================
# 1. PyTorch / TensorFlow GPU 확인
# ============================================================

try:
    import torch
    import torchvision
    print("[PyTorch]")
    print("torch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception as e:
    torch = None
    torch_device = None
    print("PyTorch를 불러오지 못했습니다:", e)

try:
    import tensorflow as tf
    import tensorflow_datasets as tfds
    print("\n[TensorFlow]")
    print("tensorflow:", tf.__version__)
    print("GPUs:", tf.config.list_physical_devices("GPU"))
except Exception as e:
    tf = None
    print("TensorFlow를 불러오지 못했습니다:", e)

## 1. 데이터셋 준비

STL-10은 10개의 클래스를 가지는 RGB 이미지 데이터셋이다.  
원본 이미지는 96×96 크기이지만, 직접 구현 CNN까지 같은 조건에서 돌려보기 위해 여기서는 96×96로 resize해서 사용하였다. 이 부분은 실행 시간과 메모리 사용량을 고려한 선택이다.

학습 데이터 5000장은 과제 조건에 맞게 8:2로 나누어, 4000장은 학습용, 1000장은 검증용으로 사용한다.

In [ ]:
# ============================================================
# 2. STL-10 데이터셋 불러오기
# ============================================================

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

DATA_DIR = "./data"

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_train_dataset = datasets.STL10(
    root=DATA_DIR,
    split="train",
    download=True,
    transform=base_transform
)

test_dataset = datasets.STL10(
    root=DATA_DIR,
    split="test",
    download=True,
    transform=base_transform
)

class_names = full_train_dataset.classes

print("전체 학습 데이터:", len(full_train_dataset))
print("시험 데이터:", len(test_dataset))
print("클래스:", class_names)

In [ ]:
# ============================================================
# 3. 학습/검증 분리
# ============================================================

num_train_total = len(full_train_dataset)
indices = np.arange(num_train_total)
np.random.shuffle(indices)

train_count = int(num_train_total * 0.8)
train_idx = indices[:train_count]
val_idx = indices[train_count:]

train_dataset = Subset(full_train_dataset, train_idx)
val_dataset = Subset(full_train_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("학습 데이터:", len(train_dataset))
print("검증 데이터:", len(val_dataset))
print("시험 데이터:", len(test_dataset))

In [ ]:
# ============================================================
# 4. 이미지 확인
# ============================================================

def denormalize_torch(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.detach().cpu().numpy().transpose(1, 2, 0)
    img = img * std + mean
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(denormalize_torch(images[i]))
    plt.title(class_names[int(labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. 직접 구현 CNN

17강 코드에서 중요한 부분은 `im2col`을 이용해서 합성곱 연산을 행렬 곱으로 바꾸는 것이다.  
이번 모델도 그 구조를 그대로 사용하였다. 다만 STL-10은 컬러 이미지이므로 입력 채널 수를 3으로 바꾸고, 이미지 크기도 더 크기 때문에 convolution 계층을 3개로 늘렸다.

사용한 구조는 다음과 같다.

`Conv - ReLU - Pool - Conv - ReLU - Pool - Conv - ReLU - Pool - Affine - ReLU - Affine - Softmax`

이 구조는 pooling을 거치면서 공간 크기를 줄이고, 뒤쪽 convolution 계층으로 갈수록 더 복잡한 특징을 잡도록 만든 형태이다.

In [ ]:
# ============================================================
# 직접 구현 CNN에 사용할 NumPy 데이터 만들기
# ============================================================

def loader_to_numpy(loader):
    xs, ys = [], []
    for x, y in loader:
        xs.append(x.numpy().astype(np.float32))
        ys.append(y.numpy().astype(np.int64))
    return np.concatenate(xs, axis=0), np.concatenate(ys, axis=0)

x_train_np, t_train_np = loader_to_numpy(DataLoader(train_dataset, batch_size=256, shuffle=False))
x_val_np, t_val_np = loader_to_numpy(DataLoader(val_dataset, batch_size=256, shuffle=False))
x_test_np, t_test_np = loader_to_numpy(DataLoader(test_dataset, batch_size=256, shuffle=False))

print("x_train_np:", x_train_np.shape, t_train_np.shape)
print("x_val_np:", x_val_np.shape, t_val_np.shape)
print("x_test_np:", x_test_np.shape, t_test_np.shape)

In [ ]:
# ============================================================
# 6.  im2col / col2im
# ============================================================

def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_data.shape
    out_h = (H + 2 * pad - filter_h) // stride + 1
    out_w = (W + 2 * pad - filter_w) // stride + 1

    img = np.pad(input_data,
                 [(0, 0), (0, 0), (pad, pad), (pad, pad)],
                 mode="constant")
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w), dtype=input_data.dtype)

    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N * out_h * out_w, -1)
    return col


def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_shape
    out_h = (H + 2 * pad - filter_h) // stride + 1
    out_w = (W + 2 * pad - filter_w) // stride + 1

    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w)
    col = col.transpose(0, 3, 4, 5, 1, 2)

    img = np.zeros((N, C, H + 2 * pad + stride - 1, W + 2 * pad + stride - 1),
                   dtype=col.dtype)

    # 겹치는 위치가 있기 때문에 '='가 아니라 '+='로 누적해야 역전파가 맞다.
    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]

    return img[:, :, pad:H + pad, pad:W + pad]

In [ ]:
# ============================================================
# 7. 기본 계층 구현
# ============================================================

class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        dout = dout.copy()
        dout[self.mask] = 0
        return dout


class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.original_x_shape = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.original_x_shape = x.shape
        self.x = x.reshape(x.shape[0], -1)
        out = np.dot(self.x, self.W) + self.b
        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx.reshape(*self.original_x_shape)


class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None
        self.t = None

    def softmax(self, x):
        if x.ndim == 2:
            x = x - np.max(x, axis=1, keepdims=True)
            exp_x = np.exp(x)
            return exp_x / np.sum(exp_x, axis=1, keepdims=True)

        x = x - np.max(x)
        return np.exp(x) / np.sum(np.exp(x))

    def cross_entropy_error(self, y, t):
        if y.ndim == 1:
            y = y.reshape(1, -1)
            t = t.reshape(1, -1)

        if t.size == y.size:
            t = np.argmax(t, axis=1)

        batch_size = y.shape[0]
        return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

    def forward(self, x, t):
        self.t = t
        self.y = self.softmax(x)
        self.loss = self.cross_entropy_error(self.y, self.t)
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]

        if self.t.size == self.y.size:
            dx = (self.y - self.t) / batch_size
        else:
            dx = self.y.copy()
            dx[np.arange(batch_size), self.t] -= 1
            dx = dx / batch_size

        return dx

In [ ]:
# ============================================================
# 8. Convolution / Pooling 계층
# ============================================================

class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad

        self.x = None
        self.col = None
        self.col_W = None
        self.dW = None
        self.db = None

    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape

        out_h = 1 + (H + 2 * self.pad - FH) // self.stride
        out_w = 1 + (W + 2 * self.pad - FW) // self.stride

        col = im2col(x, FH, FW, self.stride, self.pad)
        col_W = self.W.reshape(FN, -1).T

        out = np.dot(col, col_W) + self.b
        out = out.reshape(N, out_h, out_w, FN).transpose(0, 3, 1, 2)

        self.x = x
        self.col = col
        self.col_W = col_W

        return out

    def backward(self, dout):
        FN, C, FH, FW = self.W.shape

        dout = dout.transpose(0, 2, 3, 1).reshape(-1, FN)

        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout)
        self.dW = self.dW.transpose(1, 0).reshape(FN, C, FH, FW)

        dcol = np.dot(dout, self.col_W.T)
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

        return dx


class Pooling:
    def __init__(self, pool_h, pool_w, stride=2, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad
        self.x = None
        self.arg_max = None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = 1 + (H - self.pool_h) // self.stride
        out_w = 1 + (W - self.pool_w) // self.stride

        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h * self.pool_w)

        self.arg_max = np.argmax(col, axis=1)
        out = np.max(col, axis=1)

        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)
        self.x = x

        return out

    def backward(self, dout):
        dout = dout.transpose(0, 2, 3, 1)

        pool_size = self.pool_h * self.pool_w
        dmax = np.zeros((dout.size, pool_size), dtype=dout.dtype)
        dmax[np.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()

        dmax = dmax.reshape(dout.shape + (pool_size,))
        dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2], -1)

        dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)
        return dx

In [ ]:
# ============================================================
# 9. Optimizer
# ============================================================

class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.iter = 0
        self.m = None
        self.v = None

    def update(self, params, grads):
        if self.m is None:
            self.m, self.v = {}, {}
            for key, value in params.items():
                self.m[key] = np.zeros_like(value)
                self.v[key] = np.zeros_like(value)

        self.iter += 1
        lr_t = self.lr * np.sqrt(1.0 - self.beta2 ** self.iter) / (1.0 - self.beta1 ** self.iter)

        for key in params.keys():
            self.m[key] += (1 - self.beta1) * (grads[key] - self.m[key])
            self.v[key] += (1 - self.beta2) * (grads[key] ** 2 - self.v[key])
            params[key] -= lr_t * self.m[key] / (np.sqrt(self.v[key]) + 1e-7)

In [ ]:
# ============================================================
# 10. 직접 구현 CNN 모델
# ============================================================

class STL10NumpyCNN:
    def __init__(self, input_dim=(3, 96, 96), hidden_size=128, output_size=10, weight_init_std="he"):
        C, H, W = input_dim

        conv_channels = [16, 32, 64]
        filter_size = 3
        pad = 1
        stride = 1

        def he_scale(fan_in):
            return np.sqrt(2.0 / fan_in)

        self.params = {}

        prev_c = C
        for i, out_c in enumerate(conv_channels, start=1):
            fan_in = prev_c * filter_size * filter_size
            scale = he_scale(fan_in) if weight_init_std == "he" else weight_init_std

            self.params[f"W{i}"] = (scale * np.random.randn(out_c, prev_c, filter_size, filter_size)).astype(np.float32)
            self.params[f"b{i}"] = np.zeros(out_c, dtype=np.float32)
            prev_c = out_c

        pooled_h = H // 8
        pooled_w = W // 8
        affine_input_size = conv_channels[-1] * pooled_h * pooled_w

        self.params["W4"] = (he_scale(affine_input_size) * np.random.randn(affine_input_size, hidden_size)).astype(np.float32)
        self.params["b4"] = np.zeros(hidden_size, dtype=np.float32)
        self.params["W5"] = (0.01 * np.random.randn(hidden_size, output_size)).astype(np.float32)
        self.params["b5"] = np.zeros(output_size, dtype=np.float32)

        self.layers = OrderedDict()
        self.layers["Conv1"] = Convolution(self.params["W1"], self.params["b1"], stride=stride, pad=pad)
        self.layers["Relu1"] = Relu()
        self.layers["Pool1"] = Pooling(2, 2, stride=2)

        self.layers["Conv2"] = Convolution(self.params["W2"], self.params["b2"], stride=stride, pad=pad)
        self.layers["Relu2"] = Relu()
        self.layers["Pool2"] = Pooling(2, 2, stride=2)

        self.layers["Conv3"] = Convolution(self.params["W3"], self.params["b3"], stride=stride, pad=pad)
        self.layers["Relu3"] = Relu()
        self.layers["Pool3"] = Pooling(2, 2, stride=2)

        self.layers["Affine1"] = Affine(self.params["W4"], self.params["b4"])
        self.layers["Relu4"] = Relu()
        self.layers["Affine2"] = Affine(self.params["W5"], self.params["b5"])

        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.last_layer.forward(y, t)

    def accuracy(self, x, t, batch_size=128):
        if t.ndim != 1:
            t = np.argmax(t, axis=1)

        correct = 0
        total = x.shape[0]

        for i in range(0, total, batch_size):
            tx = x[i:i + batch_size]
            tt = t[i:i + batch_size]
            y = self.predict(tx)
            y = np.argmax(y, axis=1)
            correct += np.sum(y == tt)

        return correct / total

    def gradient(self, x, t):
        self.loss(x, t)

        dout = self.last_layer.backward(1)

        layers = list(self.layers.values())
        layers.reverse()

        for layer in layers:
            dout = layer.backward(dout)

        grads = {}
        for i in range(1, 4):
            grads[f"W{i}"] = self.layers[f"Conv{i}"].dW
            grads[f"b{i}"] = self.layers[f"Conv{i}"].db

        grads["W4"] = self.layers["Affine1"].dW
        grads["b4"] = self.layers["Affine1"].db
        grads["W5"] = self.layers["Affine2"].dW
        grads["b5"] = self.layers["Affine2"].db

        return grads

In [ ]:
# ============================================================
# 11. 파라미터 수 / FLOPs 계산 함수
# ============================================================

def count_numpy_params(model):
    return int(sum(np.prod(v.shape) for v in model.params.values()))

def estimate_cnn_flops(img_size=96):
    # Conv FLOPs는 multiply와 add를 각각 1번으로 보고 2를 곱했다.
    flops = 0


    flops += 2 * img_size * img_size * 16 * 3 * 3 * 3

 
    s = img_size // 2
    flops += 2 * s * s * 32 * 16 * 3 * 3

    s = s // 2
    flops += 2 * s * s * 64 * 32 * 3 * 3

    s = s // 2
    affine_in = 64 * s * s

    flops += 2 * affine_in * 128
    flops += 2 * 128 * 10

    return int(flops)

tmp_model = STL10NumpyCNN(input_dim=(3, IMG_SIZE, IMG_SIZE))
numpy_param_count = count_numpy_params(tmp_model)
model_flops = estimate_cnn_flops(IMG_SIZE)

print("직접 구현 CNN 파라미터 수:", numpy_param_count)
print("직접 구현 CNN 추정 FLOPs / image:", model_flops)

In [ ]:
# ============================================================
# 12. 직접 구현 CNN 학습
# ============================================================

from tqdm.notebook import tqdm
import time
import copy

def train_numpy_cnn(model, x_train, t_train, x_val, t_val,
                    epochs=50, batch_size=64, lr=0.001, eval_batch_size=128,
                    eval_train_sample=512):
    optimizer = Adam(lr=lr)

    train_loss_list = []
    train_acc_list = []
    val_acc_list = []

    best_val_acc = 0.0
    best_params = copy.deepcopy(model.params)

    train_size = x_train.shape[0]
    iter_per_epoch = max(train_size // batch_size, 1)

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        order = np.random.permutation(train_size)

        pbar = tqdm(
            range(iter_per_epoch),
            desc=f"NumPy CNN Epoch {epoch}/{epochs}",
            leave=True
        )

        for i in pbar:
            batch_idx = order[i * batch_size:(i + 1) * batch_size]
            x_batch = x_train[batch_idx]
            t_batch = t_train[batch_idx]

            batch_start = time.time()

            grads = model.gradient(x_batch, t_batch)
            optimizer.update(model.params, grads)

            loss = model.loss(x_batch, t_batch)
            epoch_loss += loss

            batch_time = time.time() - batch_start

            pbar.set_postfix({
                "batch": f"{i+1}/{iter_per_epoch}",
                "loss": f"{loss:.4f}",
                "sec/batch": f"{batch_time:.1f}"
            })

        avg_loss = epoch_loss / iter_per_epoch

        # 학습 정확도는 전체 4000장으로 매번 계산하면 너무 오래 걸려서 일부만 확인
        train_eval_size = min(eval_train_sample, x_train.shape[0])
        train_acc = model.accuracy(
            x_train[:train_eval_size],
            t_train[:train_eval_size],
            batch_size=eval_batch_size
        )

        # 검증 데이터는 1000장이라 전체 확인
        val_acc = model.accuracy(
            x_val,
            t_val,
            batch_size=eval_batch_size
        )

        train_loss_list.append(avg_loss)
        train_acc_list.append(train_acc)
        val_acc_list.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_params = copy.deepcopy(model.params)

        elapsed = time.time() - start_time

        print(
            f"[NumPy CNN] Epoch {epoch:02d}/{epochs} | "
            f"loss {avg_loss:.4f} | "
            f"train_acc(sample) {train_acc:.4f} | "
            f"val_acc {val_acc:.4f} | "
            f"elapsed {elapsed/60:.1f} min"
        )

    model.params.update(best_params)

    history = {
        "loss": train_loss_list,
        "train_acc": train_acc_list,
        "val_acc": val_acc_list,
        "best_val_acc": best_val_acc,
        "time_sec": time.time() - start_time
    }

    return history

In [ ]:

numpy_model = STL10NumpyCNN(
    input_dim=(3, IMG_SIZE, IMG_SIZE),
    hidden_size=128,
    output_size=10
)

numpy_history = train_numpy_cnn(
    numpy_model,
    x_train_np, t_train_np,
    x_val_np, t_val_np,
    epochs=50,
    batch_size=BATCH_SIZE,
    lr=0.001,
    eval_batch_size=128,
    eval_train_sample=512
)

numpy_test_acc = numpy_model.accuracy(
    x_test_np,
    t_test_np,
    batch_size=128
)

print("직접 구현 CNN test accuracy:", numpy_test_acc)

In [ ]:
# ============================================================
# 13. 직접 구현 CNN 학습 과정 시각화
# ============================================================

if "numpy_history" not in globals():
    print("아직 numpy_history가 없다.")
else:
    epochs_range = np.arange(1, len(numpy_history["loss"]) + 1)

    train_loss = np.array(numpy_history["loss"])
    train_acc = np.array(numpy_history["train_acc"])
    val_acc = np.array(numpy_history["val_acc"])

    best_epoch = int(np.argmax(val_acc)) + 1
    best_val_acc = float(np.max(val_acc))

    # 1) Validation Accuracy 중심 그래프
    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs_range,
        val_acc,
        marker="o",
        markersize=4,
        linewidth=2.5,
        label="Validation Accuracy"
    )
    plt.axvline(
        best_epoch,
        linestyle="--",
        linewidth=2,
        label=f"Best Epoch: {best_epoch}"
    )
    plt.scatter(best_epoch, best_val_acc, s=140, zorder=5)

    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.title("NumPy CNN Validation Accuracy", fontsize=16, fontweight="bold")
    plt.ylim(0.35, 0.65)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # 2) Train / Validation Accuracy 비교
    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs_range,
        train_acc,
        linewidth=2.5,
        label="Train Accuracy"
    )
    plt.plot(
        epochs_range,
        val_acc,
        linewidth=2.5,
        label="Validation Accuracy"
    )
    plt.axvline(
        best_epoch,
        linestyle="--",
        linewidth=2,
        label=f"Best Val Epoch: {best_epoch}"
    )

    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.title("NumPy CNN Accuracy", fontsize=16, fontweight="bold")
    plt.ylim(0.3, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # 3) Training Loss는 로그 스케일로 표시
    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs_range,
        train_loss,
        marker="o",
        markersize=4,
        linewidth=2.5,
        label="Training Loss"
    )

    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss")
    plt.title("NumPy CNN Training Loss (Log Scale)", fontsize=16, fontweight="bold")
    plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

## 3. PyTorch로 같은 모델 구현

직접 구현 모델과 비교하기 위해 PyTorch에서도 같은 순서의 계층을 사용하였다.  
차이는 직접 구현 모델은 `im2col`과 직접 작성한 역전파를 사용하고, PyTorch는 autograd가 기울기 계산을 자동으로 처리한다는 점이다.

In [ ]:
# ============================================================
# 14. PyTorch 모델
# ============================================================

import torch.nn as nn
import torch.optim as optim

class TorchSTL10CNN(nn.Module):
    def __init__(self, num_classes=10, img_size=96):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        final_size = img_size // 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * final_size * final_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


torch_model = TorchSTL10CNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(torch_device)

torch_param_count = sum(p.numel() for p in torch_model.parameters())
print(torch_model)
print("PyTorch CNN 파라미터 수:", torch_param_count)
print("추정 FLOPs / image:", model_flops)

In [ ]:
# ============================================================
# 15. PyTorch 학습 / 평가 함수
# ============================================================

def train_one_epoch_torch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (pred.argmax(dim=1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_torch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        pred = model(x)
        loss = criterion(pred, y)

        total_loss += loss.item() * x.size(0)
        correct += (pred.argmax(dim=1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

In [ ]:
# ============================================================
# 16. PyTorch 학습 실행
# ============================================================

set_seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

torch_model = TorchSTL10CNN(num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(torch_device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(torch_model.parameters(), lr=0.001)

torch_history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_acc = 0.0
best_torch_state = copy.deepcopy(torch_model.state_dict())

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch_torch(torch_model, train_loader, criterion, optimizer, torch_device)
    val_loss, val_acc = evaluate_torch(torch_model, val_loader, criterion, torch_device)

    torch_history["train_loss"].append(train_loss)
    torch_history["val_loss"].append(val_loss)
    torch_history["train_acc"].append(train_acc)
    torch_history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_torch_state = copy.deepcopy(torch_model.state_dict())

    print(f"[PyTorch CNN] Epoch {epoch:02d}/{EPOCHS} | train_acc {train_acc:.4f} | val_acc {val_acc:.4f}")

torch_time = time.time() - start_time
torch_model.load_state_dict(best_torch_state)

test_loss, torch_test_acc = evaluate_torch(torch_model, test_loader, criterion, torch_device)

print("PyTorch CNN test accuracy:", torch_test_acc)
print("PyTorch training time(sec):", torch_time)

In [ ]:
epochs_range = np.arange(1, len(torch_history["train_loss"]) + 1)

torch_train_loss = np.array(torch_history["train_loss"])
torch_val_loss = np.array(torch_history["val_loss"])
torch_train_acc = np.array(torch_history["train_acc"])
torch_val_acc = np.array(torch_history["val_acc"])

# 정확도 기준 best epoch
best_acc_epoch = int(np.argmax(torch_val_acc)) + 1
best_val_acc = float(np.max(torch_val_acc))

# loss 기준 best epoch
best_loss_epoch = int(np.argmin(torch_val_loss)) + 1
best_val_loss = float(np.min(torch_val_loss))

# 1) Validation Accuracy 중심 그래프
plt.figure(figsize=(10, 5))
plt.plot(
    epochs_range,
    torch_val_acc,
    marker="o",
    markersize=4,
    linewidth=2.5,
    label="Validation Accuracy"
)
plt.axvline(
    best_acc_epoch,
    linestyle="--",
    linewidth=2,
    label=f"Best Val Acc Epoch: {best_acc_epoch}"
)
plt.scatter(best_acc_epoch, best_val_acc, s=140, zorder=5)

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("PyTorch CNN Validation Accuracy", fontsize=16, fontweight="bold")
plt.ylim(0.35, 0.70)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


# 2) Train / Validation Accuracy 비교
plt.figure(figsize=(10, 5))
plt.plot(
    epochs_range,
    torch_train_acc,
    linewidth=2.5,
    label="Train Accuracy"
)
plt.plot(
    epochs_range,
    torch_val_acc,
    linewidth=2.5,
    label="Validation Accuracy"
)
plt.axvline(
    best_acc_epoch,
    linestyle="--",
    linewidth=2,
    label=f"Best Val Acc Epoch: {best_acc_epoch}"
)

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("PyTorch CNN Train / Validation Accuracy", fontsize=16, fontweight="bold")
plt.ylim(0.25, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


# 3) Validation Loss 중심 그래프
plt.figure(figsize=(10, 5))
plt.plot(
    epochs_range,
    torch_val_loss,
    marker="o",
    markersize=4,
    linewidth=2.5,
    label="Validation Loss"
)
plt.axvline(
    best_loss_epoch,
    linestyle="--",
    linewidth=2,
    label=f"Lowest Val Loss Epoch: {best_loss_epoch}"
)
plt.scatter(best_loss_epoch, best_val_loss, s=140, zorder=5)

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("PyTorch CNN Validation Loss", fontsize=16, fontweight="bold")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_acc_epoch}")
print(f"Lowest validation loss: {best_val_loss:.4f} at epoch {best_loss_epoch}")

## 4. TensorFlow로 같은 모델 구현

TensorFlow에서도 PyTorch와 같은 구조가 되도록 구성하였다.  
TensorFlow는 기본적으로 이미지 텐서를 `(높이, 너비, 채널)` 순서로 다루므로, PyTorch용 데이터 `(채널, 높이, 너비)`를 TensorFlow 입력 형식으로 바꾸어 사용하였다.

In [ ]:
# ============================================================
# 18. TensorFlow용 데이터 변환
# ============================================================

def nchw_to_nhwc(x):
    return np.transpose(x, (0, 2, 3, 1)).astype(np.float32)

x_train_tf = nchw_to_nhwc(x_train_np)
x_val_tf = nchw_to_nhwc(x_val_np)
x_test_tf = nchw_to_nhwc(x_test_np)

print("x_train_tf:", x_train_tf.shape)
print("x_val_tf:", x_val_tf.shape)
print("x_test_tf:", x_test_tf.shape)

In [ ]:
# ============================================================
# 19. TensorFlow 모델
# ============================================================

if tf is not None:
    tf.random.set_seed(SEED)

    tf_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2, strides=2),

        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2, strides=2),

        tf.keras.layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2, strides=2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(NUM_CLASSES)
    ])

    tf_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    tf_model.summary()

    tf_param_count = tf_model.count_params()
    print("TensorFlow CNN 파라미터 수:", tf_param_count)
    print("추정 FLOPs / image:", model_flops)

In [ ]:
# ============================================================
# 20. TensorFlow 학습 실행
# ============================================================

if tf is not None:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath="best_tf_stl10_cnn.weights.h5",
            save_weights_only=True,
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=0
        )
    ]

    start_time = time.time()

    tf_history_obj = tf_model.fit(
        x_train_tf,
        t_train_np,
        validation_data=(x_val_tf, t_val_np),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    tf_time = time.time() - start_time

    tf_model.load_weights("best_tf_stl10_cnn.weights.h5")
    tf_test_loss, tf_test_acc = tf_model.evaluate(x_test_tf, t_test_np, batch_size=BATCH_SIZE, verbose=0)

    tf_history = tf_history_obj.history

    print("TensorFlow CNN test accuracy:", tf_test_acc)
    print("TensorFlow training time(sec):", tf_time)

In [ ]:
# ============================================================
# 21. TensorFlow 학습 과정 시각화
# ============================================================

if tf is not None:
    tf_train_loss = tf_history["loss"]
    tf_val_loss = tf_history["val_loss"]
    tf_train_acc = tf_history["accuracy"]
    tf_val_acc = tf_history["val_accuracy"]

    epochs_range = np.arange(1, len(tf_train_loss) + 1)

    best_epoch = int(np.argmax(tf_val_acc)) + 1
    best_val_acc = max(tf_val_acc)
    best_val_loss = tf_val_loss[best_epoch - 1]

    # 1) Validation Accuracy 중심 그래프
    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs_range,
        tf_val_acc,
        marker="o",
        markersize=4,
        linewidth=2,
        label="Validation Accuracy"
    )
    plt.axvline(
        best_epoch,
        linestyle="--",
        linewidth=2,
        label=f"Best Epoch: {best_epoch}"
    )
    plt.scatter(best_epoch, best_val_acc, s=120, zorder=5)

    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.title("TensorFlow CNN Validation Accuracy", fontsize=16, fontweight="bold")
    plt.ylim(0.35, 0.65)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # 2) Train / Validation Accuracy 비교
    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs_range,
        tf_train_acc,
        linewidth=2,
        label="Train Accuracy"
    )
    plt.plot(
        epochs_range,
        tf_val_acc,
        linewidth=2,
        label="Validation Accuracy"
    )
    plt.axvline(
        best_epoch,
        linestyle="--",
        linewidth=2,
        label=f"Best Val Epoch: {best_epoch}"
    )

    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.title("TensorFlow CNN Accuracy", fontsize=16, fontweight="bold")
    plt.ylim(0.25, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # 3) Validation Loss 중심 그래프
    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs_range,
        tf_val_loss,
        marker="o",
        markersize=4,
        linewidth=2,
        label="Validation Loss"
    )
    plt.axvline(
        best_epoch,
        linestyle="--",
        linewidth=2,
        label=f"Best Epoch: {best_epoch}"
    )
    plt.scatter(best_epoch, best_val_loss, s=120, zorder=5)

    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.title("TensorFlow CNN Validation Loss", fontsize=16, fontweight="bold")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

## 5. 시험 데이터 10개 샘플 분류 결과 확인

모델이 실제로 어떤 이미지를 맞히고 틀리는지 확인하기 위해 시험 데이터에서 10개를 골라 예측 결과를 시각화하였다.  
아래 그림에서는 정답과 예측값을 같이 표시하였다.

In [ ]:
# ============================================================
# 22. 시험 이미지 10개 분류 결과
# ============================================================

@torch.no_grad()
def show_three_model_predictions(
    numpy_model,
    torch_model,
    tf_model,
    test_dataset,
    device,
    count=10
):
    torch_model.eval()

    sample_loader = DataLoader(test_dataset, batch_size=count, shuffle=True)
    x_torch, y = next(iter(sample_loader))

    # PyTorch 예측
    torch_logits = torch_model(x_torch.to(device))
    torch_pred = torch_logits.argmax(dim=1).cpu().numpy()

    # NumPy 예측
    x_np = x_torch.numpy().astype(np.float32)
    numpy_logits = numpy_model.predict(x_np)
    numpy_pred = np.argmax(numpy_logits, axis=1)

    # TensorFlow 예측
    x_tf = np.transpose(x_np, (0, 2, 3, 1))
    tf_logits = tf_model.predict(x_tf, verbose=0)
    tf_pred = np.argmax(tf_logits, axis=1)

    y_np = y.numpy()

    fig, axes = plt.subplots(2, 5, figsize=(24, 13))

    for i, ax in enumerate(axes.flat):
        ax.imshow(denormalize_torch(x_torch[i]))

        true_name = class_names[int(y_np[i])]
        numpy_name = class_names[int(numpy_pred[i])]
        torch_name = class_names[int(torch_pred[i])]
        tf_name = class_names[int(tf_pred[i])]

        ax.set_title(
            f"True: {true_name}\n"
            f"NumPy: {numpy_name}\n"
            f"PyTorch: {torch_name}\n"
            f"TensorFlow: {tf_name}",
            fontsize=14,
            fontweight="bold",
            pad=18
        )
        ax.axis("off")

    fig.suptitle(
        "Prediction Results on 10 Test Images",
        fontsize=24,
        fontweight="bold",
        y=0.98
    )

    fig.subplots_adjust(
        top=0.86,
        hspace=0.85,
        wspace=0.18
    )

    plt.show()


show_three_model_predictions(
    numpy_model=numpy_model,
    torch_model=torch_model,
    tf_model=tf_model,
    test_dataset=test_dataset,
    device=torch_device,
    count=10
)

## 6. 세 모델 비교

세 모델은 같은 계층 구조를 사용했지만, 실제 결과는 완전히 같지 않을 수 있다.  
그 이유는 가중치 초기화 방식, 연산 순서, Conv 연산 구현 방식, GPU 연산의 비결정성, 데이터 셔플 순서 등이 조금씩 다르기 때문이다.

완전히 동일한 결과에 가깝게 만들려면 다음 조건을 최대한 맞추어야 한다.

- 같은 입력 데이터와 같은 전처리 사용
- 같은 학습/검증 분리 인덱스 사용
- 같은 초기 가중치 사용
- 같은 optimizer 설정 사용
- 같은 batch 순서 사용
- PyTorch와 TensorFlow의 Conv 초기화 방식까지 직접 맞추기

하지만 프레임워크 내부 구현이 완전히 같지는 않기 때문에 실제 실험에서는 비슷한 경향을 보이는지 확인하는 것이 더 현실적이다.

In [ ]:
# ============================================================
# 23. 결과 비교 표
# ============================================================

results = []

results.append({
    "model": "직접 구현 NumPy CNN",
    "params": numpy_param_count,
    "flops_per_image": model_flops,
    "test_acc": float(numpy_test_acc),
    "time_sec": float(numpy_history["time_sec"])
})

results.append({
    "model": "PyTorch CNN",
    "params": int(torch_param_count),
    "flops_per_image": model_flops,
    "test_acc": float(torch_test_acc),
    "time_sec": float(torch_time)
})

if tf is not None:
    results.append({
        "model": "TensorFlow CNN",
        "params": int(tf_param_count),
        "flops_per_image": model_flops,
        "test_acc": float(tf_test_acc),
        "time_sec": float(tf_time)
    })

for row in results:
    print(row)

In [ ]:
try:
    import pandas as pd
    result_df = pd.DataFrame(results)
    display(result_df)
except Exception:
    pass

## 7. AI 사용 과제: Convolution 계층별 특징맵 시각화

이 부분에서는 학습된 PyTorch 모델을 사용하였다.  
시험 이미지 100장을 통과시킨 뒤, 각 convolution 계층에서 나온 특징맵을 평균 내서 확인하였다.

초기 convolution 계층은 주로 색 변화, 경계선, 단순한 방향성 같은 낮은 수준의 특징을 잡고, 뒤쪽 계층으로 갈수록 여러 부분이 조합된 더 추상적인 특징을 잡는 경향이 있다.

In [ ]:
# ============================================================
# 24. PyTorch Conv 계층 특징맵 추출
# ============================================================

activation_maps = {}

def make_hook(name):
    def hook(module, input, output):
        activation_maps[name] = output.detach().cpu()
    return hook

hooks = []
conv_id = 1

for layer in torch_model.features:
    if isinstance(layer, nn.Conv2d):
        hooks.append(layer.register_forward_hook(make_hook(f"Conv{conv_id}")))
        conv_id += 1

# 시험 이미지 100개 선택
feature_loader = DataLoader(test_dataset, batch_size=100, shuffle=True)
feature_images, feature_labels = next(iter(feature_loader))

torch_model.eval()
with torch.no_grad():
    _ = torch_model(feature_images.to(torch_device))

for h in hooks:
    h.remove()

for name, act in activation_maps.items():
    print(name, act.shape)

In [ ]:
# ============================================================
# 25. 계층별 평균 특징맵 그리드 시각화
# ============================================================

def plot_mean_feature_maps(activation, layer_name, max_channels=16):
    # activation shape: (N, C, H, W)
    # 100개 이미지에 대해 각 채널의 평균 절댓값 특징맵을 그림
    fmap = activation.abs().mean(dim=0)  # (C, H, W)

    channel_count = min(max_channels, fmap.shape[0])
    cols = 4
    rows = int(np.ceil(channel_count / cols))

    plt.figure(figsize=(cols * 3, rows * 3))
    for i in range(channel_count):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(fmap[i], cmap="gray")
        plt.title(f"{layer_name} - ch {i}")
        plt.axis("off")

    plt.suptitle(f"{layer_name} mean feature maps from 100 test images")
    plt.tight_layout()
    plt.show()

for name, act in activation_maps.items():
    plot_mean_feature_maps(act, name, max_channels=16)

## 8. 특징맵 분석 정리

Conv1은 이미지의 가장 앞부분에서 바로 특징을 추출하므로, 색의 변화나 간단한 경계선처럼 비교적 단순한 특징이 많이 나타난다.  
Conv2는 한 번 pooling을 거친 뒤의 특징을 다시 조합하므로, 단순한 선보다 조금 더 넓은 영역의 모양이 반응한다.  
Conv3은 앞 계층의 결과를 다시 조합하기 때문에, 특정 물체의 부분적인 형태나 질감처럼 더 추상적인 특징이 나타나는 경향을 보인다.

실험 결과 세 모델의 성능이 완전히 같지는 않았다. 직접 구현 모델은 NumPy로 직접 forward/backward를 계산하기 때문에 속도가 느리고, PyTorch와 TensorFlow는 내부적으로 최적화된 Conv 연산과 자동 미분을 사용한다. 또한 세 모델의 초기화 방식과 연산 구현이 완전히 같지 않아서 같은 구조라도 결과가 조금씩 달라질 수 있다.

In [ ]:
class TorchSTL10CNN_Regularized(nn.Module):
    def __init__(self, num_classes=10, img_size=96):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        final_size = img_size // 8

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * final_size * final_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
train_transform_aug = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

aug_full_train_dataset = datasets.STL10(
    root=DATA_DIR,
    split="train",
    download=False,
    transform=train_transform_aug
)

train_dataset_aug = Subset(aug_full_train_dataset, train_idx)

train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

print("PyTorch 증강 학습 데이터:", len(train_dataset_aug))

In [ ]:
# PyTorch 과적합 완화 학습 실행
set_seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

torch_model = TorchSTL10CNN_Regularized(
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE
).to(torch_device)

criterion = nn.CrossEntropyLoss()

# weight_decay가 L2 regularization 역할을 한다.
optimizer = optim.Adam(
    torch_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# 검증 loss가 좋아지지 않으면 learning rate를 줄인다.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

torch_history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_loss = float("inf")
best_val_acc = 0.0
best_torch_state = copy.deepcopy(torch_model.state_dict())

patience = 8
early_stop_count = 0

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch_torch(
        torch_model,
        train_loader_aug,
        criterion,
        optimizer,
        torch_device
    )

    val_loss, val_acc = evaluate_torch(
        torch_model,
        val_loader,
        criterion,
        torch_device
    )

    scheduler.step(val_loss)

    torch_history["train_loss"].append(train_loss)
    torch_history["val_loss"].append(val_loss)
    torch_history["train_acc"].append(train_acc)
    torch_history["val_acc"].append(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"[PyTorch Regularized Epoch {epoch:02d}/{EPOCHS}] "
        f"lr={current_lr:.6f} | "
        f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f} | "
        f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}"
    )

    # validation loss 기준으로 best model 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_torch_state = copy.deepcopy(torch_model.state_dict())
        early_stop_count = 0
    else:
        early_stop_count += 1

    if early_stop_count >= patience:
        print(f"Early stopping 발생: {epoch} epoch에서 학습 종료")
        break

torch_time = time.time() - start_time

torch_model.load_state_dict(best_torch_state)

test_loss, torch_test_acc = evaluate_torch(
    torch_model,
    test_loader,
    criterion,
    torch_device
)

print("PyTorch regularized test accuracy:", torch_test_acc)
print("PyTorch regularized training time(sec):", torch_time)

In [ ]:
def prepare_tf_input_fixed(x):
    x = np.array(x).astype(np.float32)

    # PyTorch 형식: (N, C, H, W) 이면 TensorFlow 형식으로 변경
    if x.ndim == 4 and x.shape[1] == 3:
        x = np.transpose(x, (0, 2, 3, 1))

    # ImageNet Normalize가 적용된 값이면 되돌리기
    # PyTorch Normalize 후에는 보통 min이 음수, max가 1보다 큼
    if x.min() < 0:
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

        x = x * std + mean
        x = np.clip(x, 0, 1)

    # 원본 이미지가 0~255 범위이면 0~1로 변경
    elif x.max() > 1.5:
        x = x / 255.0
        x = np.clip(x, 0, 1)

    # 이미 0~1이면 그대로 사용
    else:
        x = np.clip(x, 0, 1)

    return x.astype(np.float32)


x_train_tf_fixed = prepare_tf_input_fixed(x_train_tf)
x_val_tf_fixed = prepare_tf_input_fixed(x_val_tf)
x_test_tf_fixed = prepare_tf_input_fixed(x_test_tf)

print("x_train_tf_fixed shape:", x_train_tf_fixed.shape)
print("x_val_tf_fixed shape:", x_val_tf_fixed.shape)
print("x_test_tf_fixed shape:", x_test_tf_fixed.shape)
print("train min/max:", x_train_tf_fixed.min(), x_train_tf_fixed.max())
print("val min/max:", x_val_tf_fixed.min(), x_val_tf_fixed.max())
print("test min/max:", x_test_tf_fixed.min(), x_test_tf_fixed.max())

In [ ]:
if tf is not None:
    tf.random.set_seed(SEED)

    l2_reg = tf.keras.regularizers.l2(1e-6)

    tf_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        # 약한 데이터 증강
        tf.keras.layers.RandomFlip("horizontal"),

        # Block 1
        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.MaxPooling2D(pool_size=2),

        # Block 2
        tf.keras.layers.Conv2D(64, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.Conv2D(64, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Dropout(0.15),

        # Block 3
        tf.keras.layers.Conv2D(128, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.Conv2D(128, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Dropout(0.20),

        # Block 4
        tf.keras.layers.Conv2D(256, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),

        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Dropout(0.25),

        # 여기서 GlobalAveragePooling 대신 Flatten 사용
        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            256,
            activation="relu",
            kernel_regularizer=l2_reg
        ),
        tf.keras.layers.Dropout(0.30),

        tf.keras.layers.Dense(NUM_CLASSES)
    ])

    tf_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    tf_model.summary()

    tf_param_count = tf_model.count_params()
    print("TensorFlow CNN 버전 파라미터 수:", tf_param_count)

In [ ]:
if tf is not None:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath="best_tf_stl10_cnn_fixed.weights.h5",
            save_weights_only=True,
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

    start_time = time.time()

    tf_history_obj = tf_model.fit(
        x_train_tf_fixed,
        t_train_np,
        validation_data=(x_val_tf_fixed, t_val_np),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    tf_time = time.time() - start_time

    tf_model.load_weights("best_tf_stl10_cnn_fixed.weights.h5")

    tf_test_loss, tf_test_acc = tf_model.evaluate(
        x_test_tf_fixed,
        t_test_np,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    tf_history = tf_history_obj.history

    print("TensorFlow fixed test loss:", tf_test_loss)
    print("TensorFlow fixed test accuracy:", tf_test_acc)
    print("TensorFlow fixed training time(sec):", tf_time)

In [ ]:
# ============================================================
# PyTorch / TensorFlow 과적합 확인
# ============================================================

def plot_history(history, title, train_acc_key, val_acc_key, train_loss_key, val_loss_key):
    epochs_range = np.arange(1, len(history[train_loss_key]) + 1)

    train_loss = np.array(history[train_loss_key])
    val_loss = np.array(history[val_loss_key])
    train_acc = np.array(history[train_acc_key])
    val_acc = np.array(history[val_acc_key])

    best_epoch = int(np.argmax(val_acc)) + 1
    best_val_acc = float(np.max(val_acc))

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, train_acc, marker="o", label="Train Accuracy")
    plt.plot(epochs_range, val_acc, marker="o", label="Validation Accuracy")
    plt.axvline(best_epoch, linestyle="--", label=f"Best Epoch: {best_epoch}")
    plt.scatter(best_epoch, best_val_acc, s=120, zorder=5)

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} Accuracy")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, train_loss, marker="o", label="Train Loss")
    plt.plot(epochs_range, val_loss, marker="o", label="Validation Loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# PyTorch 그래프
plot_history(
    torch_history,
    title="PyTorch Regularized CNN",
    train_acc_key="train_acc",
    val_acc_key="val_acc",
    train_loss_key="train_loss",
    val_loss_key="val_loss"
)

# TensorFlow 그래프
if tf is not None:
    plot_history(
        tf_history,
        title="TensorFlow Regularized CNN",
        train_acc_key="accuracy",
        val_acc_key="val_accuracy",
        train_loss_key="loss",
        val_loss_key="val_loss"
    )

In [ ]:
@torch.no_grad()
def show_torch_tf_predictions(
    torch_model,
    tf_model,
    test_dataset,
    device,
    count=10
):
    torch_model.eval()

    # STL-10 클래스 이름
    class_names = [
        "airplane", "bird", "car", "cat", "deer",
        "dog", "horse", "monkey", "ship", "truck"
    ]

    # 같은 이미지 count개를 두 모델에 모두 넣기 위해 한 번만 샘플링
    sample_loader = DataLoader(
        test_dataset,
        batch_size=count,
        shuffle=True
    )

    x_torch, y = next(iter(sample_loader))

    # ------------------------------------------------------------
    # 1) PyTorch 모델 예측
    # PyTorch 모델은 Normalize된 입력을 그대로 사용
    # ------------------------------------------------------------
    torch_logits = torch_model(x_torch.to(device))
    torch_pred = torch_logits.argmax(dim=1).cpu().numpy()

    # ------------------------------------------------------------
    # 2) PyTorch 입력을 numpy로 변환
    # x_np는 현재 Normalize된 상태
    # shape: (N, C, H, W)
    # ------------------------------------------------------------
    x_np = x_torch.cpu().numpy().astype(np.float32)

    # ------------------------------------------------------------
    # 3) 이미지 시각화 및 TensorFlow 입력을 위해 Normalize 되돌리기
    # TensorFlow는 0~1 범위의 (N, H, W, C) 이미지를 사용
    # ------------------------------------------------------------
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    images = np.transpose(x_np, (0, 2, 3, 1))
    images = images * std + mean
    images = np.clip(images, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 4) TensorFlow 모델 예측
    # 중요: Normalize된 x_tf가 아니라 Normalize를 되돌린 images 사용
    # ------------------------------------------------------------
    tf_logits = tf_model.predict(images, verbose=0)
    tf_pred = np.argmax(tf_logits, axis=1)

    y_np = y.cpu().numpy()

    # ------------------------------------------------------------
    # 5) 결과 출력
    # ------------------------------------------------------------
    plt.figure(figsize=(22, count * 2.4))

    for i in range(count):
        true_label = y_np[i]
        torch_label = torch_pred[i]
        tf_label = tf_pred[i]

        torch_correct = torch_label == true_label
        tf_correct = tf_label == true_label

        torch_mark = "TRUE" if torch_correct else "FALSE"
        tf_mark = "TRUE" if tf_correct else "FALSE"

        plt.subplot(count, 1, i + 1)
        plt.imshow(images[i])
        plt.axis("off")

        title = (
            f"True: {class_names[true_label]}  |  "
            f"PyTorch: {class_names[torch_label]} ({torch_mark})  |  "
            f"TensorFlow: {class_names[tf_label]} ({tf_mark})"
        )

        plt.title(title, fontsize=18, pad=18)

    plt.subplots_adjust(hspace=0.75)
    plt.show()


show_torch_tf_predictions(
    torch_model=torch_model,
    tf_model=tf_model,
    test_dataset=test_dataset,
    device=torch_device,
    count=10
)